# Trading Simulation: Can Our Models Beat the Market?

This notebook backtests a simple value-betting strategy on the test split.

**Polymarket token economics**  
A YES token costs `price_at_snapshot` and pays $1 if the market resolves YES, $0 otherwise.  
A NO token costs `1 - price_at_snapshot` and pays $1 if the market resolves NO.

**Strategy**  
For each snapshot, compare the model's P(YES) to the current market price:
- `model_prob > market_price + edge`: buy YES → profit = `outcome - market_price`
- `model_prob < market_price - edge`: buy NO → profit = `(1 - outcome) - (1 - market_price)` = `market_price - outcome`
- Otherwise: no trade (0 profit)

**Baseline**  
The market-price baseline (AUC 0.964) essentially *is* the market — it never disagrees with itself, so it never trades and earns 0.  
A naïve always-buy-YES strategy earns `mean(outcome) - mean(market_price)` ≈ 0 on an efficient market.

We report **total P&L** and **ROI** (P&L / capital deployed) per model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

DATA_DIR   = '../data/'
MODELS_DIR = '../models/'

# Minimum edge (model_prob vs market_price) required to place a trade
MIN_EDGE = 0.02

## 1. Load Data

In [ ]:
# Load only the columns we need from the clean dataset (test split)
cols = ['market_id', 'snapshot_timestamp', 'price_at_snapshot', 'outcome',
        'split', 'category', 'days_before_close']

df_full = pd.read_parquet(DATA_DIR + 'polymarket_ml_dataset_clean.parquet', columns=cols)
df = df_full[df_full['split'] == 'test'].copy().reset_index(drop=True)

# Normalize timestamp timezone for merging
df['snapshot_timestamp'] = pd.to_datetime(df['snapshot_timestamp'], utc=True)

print(f"Test snapshots: {len(df):,}")
print(f"Unique markets:  {df['market_id'].nunique():,}")
print(f"Outcome balance: {df['outcome'].mean():.1%} YES")
df.head(3)

In [ ]:
# Load model predictions
lr = pd.read_csv(MODELS_DIR + 'logistic_regression/predictions/predictions.csv',
                 parse_dates=['snapshot_timestamp'])
lr['snapshot_timestamp'] = pd.to_datetime(lr['snapshot_timestamp'], utc=True)

rf = pd.read_csv(MODELS_DIR + 'random_forest/predictions/test_predictions.csv',
                 parse_dates=['snapshot_timestamp'])
rf['snapshot_timestamp'] = pd.to_datetime(rf['snapshot_timestamp'], utc=True)

gb = pd.read_csv(MODELS_DIR + 'gradient_boosting/predictions/predictions_v2.csv')

print("LR columns:", lr.columns.tolist())
print("RF columns:", rf.columns.tolist())
print("GB columns:", gb.columns.tolist())

## 2. Merge Predictions with Market Prices

LR and RF include `snapshot_timestamp` so we can join on `(market_id, snapshot_timestamp)`.

GB has no timestamp column — we aggregate per market instead (see Section 4).

In [ ]:
key = ['market_id', 'snapshot_timestamp']
price_cols = key + ['price_at_snapshot', 'category', 'days_before_close']

lr_merged = lr.merge(df[price_cols], on=key, how='left')
rf_merged = rf.merge(df[price_cols], on=key, how='left')

print(f"LR merged rows: {len(lr_merged):,}  (NaN prices: {lr_merged['price_at_snapshot'].isna().sum():,})")
print(f"RF merged rows: {len(rf_merged):,}  (NaN prices: {rf_merged['price_at_snapshot'].isna().sum():,})")

## 3. Trading Strategy Helper

In [ ]:
def simulate_trades(model_prob, market_price, outcome, min_edge=MIN_EDGE):
    """
    Returns a Series of per-trade P&L (0 = no trade).
    Assumes 1-unit stake per trade.
    """
    edge = model_prob - market_price
    pnl = np.zeros(len(model_prob))

    buy_yes = edge > min_edge
    buy_no  = edge < -min_edge

    # YES trade: pay market_price, receive outcome
    pnl[buy_yes] = outcome[buy_yes] - market_price[buy_yes]

    # NO trade: pay (1 - market_price), receive (1 - outcome)
    pnl[buy_no] = market_price[buy_no] - outcome[buy_no]

    return pd.Series(pnl)


def trading_summary(pnl_series, market_price_series, model_name):
    traded = pnl_series != 0
    n_trades    = traded.sum()
    total_pnl   = pnl_series.sum()
    capital     = market_price_series[traded].sum()   # approximate capital deployed (YES side)
    roi         = total_pnl / capital if capital > 0 else np.nan
    win_rate    = (pnl_series[traded] > 0).mean() if n_trades > 0 else np.nan

    return {
        'model':     model_name,
        'n_trades':  int(n_trades),
        'total_pnl': round(total_pnl, 2),
        'roi_%':     round(roi * 100, 2) if not np.isnan(roi) else np.nan,
        'win_rate_%': round(win_rate * 100, 2) if not np.isnan(win_rate) else np.nan,
    }

## 4. Per-Snapshot Simulation (LR & RF)

Trade at every snapshot in the test set.

In [ ]:
results = []
pnl_series = {}

# ----- Logistic Regression -----
sub = lr_merged.dropna(subset=['price_at_snapshot'])
for col, label in [('pred_prob_base', 'LR (base)'), ('pred_prob_trends', 'LR (trends)')]:
    pnl = simulate_trades(sub[col].values, sub['price_at_snapshot'].values, sub['outcome'].values)
    pnl_series[label] = pnl
    results.append(trading_summary(pnl, sub['price_at_snapshot'], label))

# ----- Random Forest -----
sub = rf_merged.dropna(subset=['price_at_snapshot'])
for col, label in [('proba_full', 'RF (full)'), ('proba_full_calibrated', 'RF (calibrated)')]:
    pnl = simulate_trades(sub[col].values, sub['price_at_snapshot'].values, sub['outcome'].values)
    pnl_series[label] = pnl
    results.append(trading_summary(pnl, sub['price_at_snapshot'], label))

summary_df = pd.DataFrame(results)
print("Per-snapshot trading results (min_edge={:.2f})".format(MIN_EDGE))
summary_df

## 5. Per-Market Simulation (LR, RF, GB)

Aggregate each model's probability across all snapshots of a market, then make one trade decision per market.
This avoids double-counting correlated snapshots.

For GB (no timestamp), we aggregate its predictions by `market_id` and join to the per-market mean market price.

In [ ]:
# Per-market mean market price and outcome from the test set
mkt = (df.groupby('market_id')
         .agg(market_price=('price_at_snapshot', 'mean'),
              outcome=('outcome', 'first'),
              category=('category', 'first'))
         .reset_index())

print(f"Unique markets: {len(mkt):,}")
mkt.head(3)

In [ ]:
pm_results = []
pm_pnl = {}

# LR per-market
lr_pm = lr_merged.groupby('market_id')[['pred_prob_base', 'pred_prob_trends']].mean().reset_index()
lr_pm = lr_pm.merge(mkt, on='market_id')

for col, label in [('pred_prob_base', 'LR (base)'), ('pred_prob_trends', 'LR (trends)')]:
    pnl = simulate_trades(lr_pm[col].values, lr_pm['market_price'].values, lr_pm['outcome'].values)
    pm_pnl[label] = (pnl, lr_pm['category'].values)
    pm_results.append(trading_summary(pnl, lr_pm['market_price'], label))

# RF per-market
rf_pm = rf_merged.groupby('market_id')[['proba_full', 'proba_full_calibrated']].mean().reset_index()
rf_pm = rf_pm.merge(mkt, on='market_id')

for col, label in [('proba_full', 'RF (full)'), ('proba_full_calibrated', 'RF (calibrated)')]:
    pnl = simulate_trades(rf_pm[col].values, rf_pm['market_price'].values, rf_pm['outcome'].values)
    pm_pnl[label] = (pnl, rf_pm['category'].values)
    pm_results.append(trading_summary(pnl, rf_pm['market_price'], label))

# GB per-market
gb_pm = gb.groupby('market_id')[['pred_prob_base_v2', 'pred_prob_base_v2_cal',
                                   'pred_prob_trends_v2', 'pred_prob_trends_v2_cal']].mean().reset_index()
gb_pm = gb_pm.merge(mkt, on='market_id')

for col, label in [('pred_prob_base_v2', 'GB (base)'),
                   ('pred_prob_base_v2_cal', 'GB (calibrated)'),
                   ('pred_prob_trends_v2', 'GB (trends)'),
                   ('pred_prob_trends_v2_cal', 'GB (trends+cal)')]:
    pnl = simulate_trades(gb_pm[col].values, gb_pm['market_price'].values, gb_pm['outcome'].values)
    pm_pnl[label] = (pnl, gb_pm['category'].values)
    pm_results.append(trading_summary(pnl, gb_pm['market_price'], label))

pm_summary = pd.DataFrame(pm_results)
print("Per-market trading results (min_edge={:.2f})".format(MIN_EDGE))
pm_summary

## 6. Naïve Baselines

In [ ]:
# Always buy YES on every market
naive_pnl = mkt['outcome'] - mkt['market_price']
print("=== Naïve always-buy-YES (per market) ===")
print(f"  Total P&L:  {naive_pnl.sum():.2f}")
print(f"  Mean P&L:   {naive_pnl.mean():.4f}  (should be ≈ 0 on an efficient market)")
print(f"  Capital:    {mkt['market_price'].sum():.2f}")
print(f"  ROI:        {naive_pnl.sum() / mkt['market_price'].sum() * 100:.2f}%")

# Market-price model → never disagrees → 0 trades
print("\n=== Market-price baseline ===")
print("  Since the baseline model predicts market_price ≈ itself, edge = 0 → 0 trades → P&L = 0")

## 7. ROI by Min-Edge Threshold

How does ROI change as we require a larger edge before trading?

In [ ]:
edges = np.arange(0.0, 0.31, 0.02)

# Focus on two representative models: LR (base) and GB (calibrated)
focus_models = {
    'LR (base)':       (lr_pm, 'pred_prob_base'),
    'RF (calibrated)': (rf_pm, 'proba_full_calibrated'),
    'GB (calibrated)': (gb_pm, 'pred_prob_base_v2_cal'),
}

edge_results = {name: [] for name in focus_models}
trade_counts = {name: [] for name in focus_models}

for e in edges:
    for name, (pm_df, col) in focus_models.items():
        pnl = simulate_trades(pm_df[col].values, pm_df['market_price'].values,
                              pm_df['outcome'].values, min_edge=e)
        traded = pnl != 0
        cap = pm_df['market_price'].values[traded.values].sum()
        roi = pnl.sum() / cap * 100 if cap > 0 else 0
        edge_results[name].append(roi)
        trade_counts[name].append(traded.sum())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for name, rois in edge_results.items():
    ax1.plot(edges, rois, marker='o', markersize=4, label=name)
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.set_xlabel('Minimum Edge Threshold')
ax1.set_ylabel('ROI (%)')
ax1.set_title('ROI vs Min-Edge Threshold (per market)')
ax1.legend()
ax1.grid(alpha=0.3)

for name, counts in trade_counts.items():
    ax2.plot(edges, counts, marker='o', markersize=4, label=name)
ax2.set_xlabel('Minimum Edge Threshold')
ax2.set_ylabel('Number of Trades')
ax2.set_title('Trade Volume vs Min-Edge Threshold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('trading_roi_vs_edge.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Cumulative P&L Over Time (LR & RF per-snapshot)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Use LR timestamps as the time axis (same test split)
sub_lr = lr_merged.dropna(subset=['price_at_snapshot']).copy()
sub_lr = sub_lr.sort_values('snapshot_timestamp')

for col, label, color in [
    ('pred_prob_base',   'LR (base)',   'steelblue'),
    ('pred_prob_trends', 'LR (trends)', 'cornflowerblue'),
]:
    pnl = simulate_trades(sub_lr[col].values, sub_lr['price_at_snapshot'].values, sub_lr['outcome'].values)
    ax.plot(sub_lr['snapshot_timestamp'].values, pnl.cumsum().values, label=label, color=color, linewidth=0.8)

sub_rf = rf_merged.dropna(subset=['price_at_snapshot']).copy()
sub_rf = sub_rf.sort_values('snapshot_timestamp')

for col, label, color in [
    ('proba_full',            'RF (full)',        'darkorange'),
    ('proba_full_calibrated', 'RF (calibrated)',  'sandybrown'),
]:
    pnl = simulate_trades(sub_rf[col].values, sub_rf['price_at_snapshot'].values, sub_rf['outcome'].values)
    ax.plot(sub_rf['snapshot_timestamp'].values, pnl.cumsum().values, label=label, color=color, linewidth=0.8)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--', label='Baseline (0)')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative P&L (units)')
ax.set_title(f'Cumulative P&L Over Time (min_edge={MIN_EDGE})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('trading_cumulative_pnl.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. P&L Breakdown by Category

In [ ]:
# Use GB (calibrated) and LR (base) per-market for category breakdown
cat_models = [
    (gb_pm,  'pred_prob_base_v2_cal', 'GB (calibrated)'),
    (lr_pm,  'pred_prob_base',        'LR (base)'),
    (rf_pm,  'proba_full_calibrated', 'RF (calibrated)'),
]

cat_rows = []
for pm_df, col, label in cat_models:
    pnl = simulate_trades(pm_df[col].values, pm_df['market_price'].values, pm_df['outcome'].values)
    tmp = pm_df[['category', 'market_price']].copy()
    tmp['pnl'] = pnl.values
    tmp['traded'] = (pnl != 0).values
    grp = tmp[tmp['traded']].groupby('category').agg(
        total_pnl=('pnl', 'sum'),
        n_trades=('pnl', 'count'),
        capital=('market_price', 'sum')
    ).reset_index()
    grp['roi_%'] = grp['total_pnl'] / grp['capital'] * 100
    grp['model'] = label
    cat_rows.append(grp)

cat_df = pd.concat(cat_rows, ignore_index=True)

fig, ax = plt.subplots(figsize=(12, 5))
categories = cat_df['category'].unique()
x = np.arange(len(categories))
width = 0.25

for i, (_, _, label) in enumerate(cat_models):
    vals = [cat_df[(cat_df['model'] == label) & (cat_df['category'] == c)]['roi_%'].values[0]
            if len(cat_df[(cat_df['model'] == label) & (cat_df['category'] == c)]) > 0
            else 0
            for c in categories]
    ax.bar(x + i * width, vals, width, label=label)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x + width)
ax.set_xticklabels(categories, rotation=30, ha='right')
ax.set_ylabel('ROI (%)')
ax.set_title(f'ROI by Category (per-market, min_edge={MIN_EDGE})')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('trading_roi_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

print(cat_df.pivot_table(index='category', columns='model', values='roi_%').round(2))

## 10. Summary Table

In [ ]:
print("=" * 60)
print(f"TRADING SIMULATION SUMMARY  (min_edge = {MIN_EDGE})")
print("=" * 60)
print("\nPer-market results:")
print(pm_summary.to_string(index=False))
print("\nPer-snapshot results (LR & RF):")
print(summary_df.to_string(index=False))
print("\nNotes:")
print("  - P&L is in Polymarket token units (1 unit = $1 token).")
print("  - ROI = total_pnl / capital_deployed (buying price of all trades).")
print("  - Market-price baseline: 0 trades, 0 P&L (by construction).")
print("  - Calibrated models tend to predict probabilities closer to market → fewer trades.")